## Topic: Convert Current in real time __Project 

### Project Overview: A Create a app that can convert Currency in Real Time

- We use the ExchangeRate-API for conversion factor

- input ---> (ExchangeRate-API) --> Conversion factor
   
    - in ExchangeRate-API:
        - (Base Currency, Target Currency)

- For this we use two tools
    - 1. first tools to hit the ExchangeRate-API and get the Conversion Factor.
        - example: 
            - Base Currency -> Bangladesh
            - Target Currency -> USA

            - here, 1 taka = 0.0081 United States Dollar
            - here, 0.0081 is the Conversion Factor


    - 2. 2nd tools give the actual result by using multiplication with Conversion
        - user query, 100 taka is equal to how much United State Dollar?

        - USA Dollar = (conversion factor * 100)
        - USA Dollar = 0.81


- Require:
    - 1. One API key from 
        - https://app.exchangerate-api.com/dashboard/confirmed
    - 2. request module
        - pip install requests



In [2]:
# Import Necessary Libraries
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

load_dotenv()

True

In [ ]:
# Create first tools 
import requests
@tool
def get_conversion_factor(base_currency: str, target_currency: str)-> float:
    """
    This Function fetches the currency conversion factor between a given base currency and a target currency
    """

    url = f"https://v6.exchangerate-api.com/v6/Your_API_KEY/pair/{base_currency}/{target_currency}"

    response = requests.get(url)
    
    # Sending data as JSON
    return response.json()



In [6]:
conversion_rate = get_conversion_factor.invoke(
    {
        "base_currency":"USD",
        "target_currency": "BDT"

    }
)

conversion_rate

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1789689601,
 'time_last_update_utc': 'Fri, 18 Sep 2026 00:00:01 +0000',
 'time_next_update_unix': 1789776001,
 'time_next_update_utc': 'Sat, 19 Sep 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'BDT',
 'conversion_rate': 123.0094}

In [20]:
# Second tool : Get the final result
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def convert(base_currency_value:int, conversion_rate:Annotated[float, InjectedToolArg])-> float:
    """
    Base on the conversion rate this function calculate the currency value by giving base currency
    """

    return base_currency_value * conversion_rate


- Here, Key note:
    - InjectedToolArg tell the Model:
        - LLM(model), do not try to fill this argument. i the developer / runtime will inject this value after running earlier tools.

In [21]:
base_currency_value = 10

USA_BDT_conversion = convert.invoke(
    {
        "base_currency_value": base_currency_value,
        "conversion_rate": 123.0094
    }
)

print(
    f"USD {base_currency_value:.2f} → "
    f"BDT {USA_BDT_conversion:.2f}"
)

USD 10.00 → BDT 1230.09


In [22]:
# --------------------------------------------------
# Create the LLM
# --------------------------------------------------

model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)


In [23]:
# --------------------------------------------------
# Bind the tool to the model
# --------------------------------------------------
model_with_tools = model.bind_tools(
    [get_conversion_factor, convert]
)



In [24]:
# --------------------------------------------------
# Ask the model a question
# --------------------------------------------------
from langchain_core.messages import HumanMessage

message = [HumanMessage("What is the conversion factor between USA BDT, and based on that can you convert 10 usa to BDT")]

message

[HumanMessage(content='What is the conversion factor between USA BDT, and based on that can you convert 10 usa to BDT', additional_kwargs={}, response_metadata={})]

In [25]:
# --------------------------------------------------
# Connect the LLM with Query (calling the tools)
# --------------------------------------------------

ai_message = model_with_tools.invoke(message)

print(ai_message)


content='' additional_kwargs={'reasoning_content': 'The user asks: "What is the conversion factor between USA BDT, and based on that can you convert 10 usa to BDT". They want conversion factor between USA (USD) and BDT (Bangladeshi Taka). They want to convert 10 USD to BDT. We need to fetch conversion factor. Use function get_conversion_factor with base_currency: "USD", target_currency: "BDT". Then use convert function with base_currency_value: 10? Wait convert function expects base_currency_value: number. But we need to pass the conversion factor? The convert function likely uses the conversion factor from the previous function. But we need to call get_conversion_factor first. Then call convert with base_currency_value: 10. The convert function will use the conversion factor internally. So we need to call get_conversion_factor first. Then call convert. Let\'s do that.', 'tool_calls': [{'id': 'fc_697a286a-7724-4bc3-82b1-5f52b3098c7d', 'function': {'arguments': '{"base_currency":"USD","

In [26]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'BDT'},
  'id': 'fc_697a286a-7724-4bc3-82b1-5f52b3098c7d',
  'type': 'tool_call'}]

In [ ]:
# merge the tools
message.append(ai_message)
message

In [31]:
for tool_call in ai_message.tool_calls:
    # execute the 1st tool and get the value of conversion rate
    if tool_call['name'] == 'get_conversion_factor':
        # execute the 1st tool
        tool_message1 = get_conversion_factor.invoke(tool_call)
        print(tool_message1)


content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1789689601, "time_last_update_utc": "Fri, 18 Sep 2026 00:00:01 +0000", "time_next_update_unix": 1789776001, "time_next_update_utc": "Sat, 19 Sep 2026 00:00:01 +0000", "base_code": "USD", "target_code": "BDT", "conversion_rate": 123.0094}' name='get_conversion_factor' tool_call_id='fc_697a286a-7724-4bc3-82b1-5f52b3098c7d'


In [ ]:
# import json

# for tool_call in ai_message.tool_calls:
#     # 1. execute the 1st tool and get the value of conversion rate
#     if tool_call['name'] == 'get_conversion_factor':
#         # execute the 1st tool
#         tool_message1 = get_conversion_factor.invoke(tool_call)
        
#         # fetch this conversion_rate
#        conversion_rate = json.loads(tool_message1.content[conversion_rate])

#         # append this tool message into message
#         message.append(tools_message1)


In [33]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    message.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    message.append(tool_message2)



In [35]:
message

[HumanMessage(content='What is the conversion factor between USA BDT, and based on that can you convert 10 usa to BDT', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "What is the conversion factor between USA BDT, and based on that can you convert 10 usa to BDT". They want conversion factor between USA (USD) and BDT (Bangladeshi Taka). They want to convert 10 USD to BDT. We need to fetch conversion factor. Use function get_conversion_factor with base_currency: "USD", target_currency: "BDT". Then use convert function with base_currency_value: 10? Wait convert function expects base_currency_value: number. But we need to pass the conversion factor? The convert function likely uses the conversion factor from the previous function. But we need to call get_conversion_factor first. Then call convert with base_currency_value: 10. The convert function will use the conversion factor internally. So we need to call get_c

In [37]:
model_with_tools.invoke(message).content

'The current conversion factor is **1\u202fUSD = 123.0094\u202fBDT**.\n\nSo, converting 10\u202fUSD:\n\n\\[\n10 \\text{\u202fUSD} \\times 123.0094 \\frac{\\text{BDT}}{\\text{USD}} = 1{,}230.094 \\text{\u202fBDT}\n\\]\n\n**10\u202fUSD ≈ 1,230.09\u202fBDT** (rounded to two decimal places).'